In [ ]:
import os
os.chdir('/content')
!pip install -q fair-esm
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/work
!cp /content/drive/MyDrive/bs6204/{train_m1.py,train_m2.py,eval_harness.py,compare_ci.py,pairs.csv,sequences.csv,wt_emb_650.pt,pairs_3d.csv,contacts.npz,m2_preds.csv,m1_650_preds.csv} /content/work/
os.chdir('/content/work')
!ls -la

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.2 MB/s eta 0:00:00
Mounted at /content/drive
total 173876
drwxr-xr-x 2 root root     4096 Oct 16 10:23 .
drwxr-xr-x 1 root root     4096 Oct 16 10:23 ..
-rw------- 1 root root     6412 Oct 16 10:23 compare_ci.py
-rw------- 1 root root   127785 Oct 16 10:23 contacts.npz
-rw------- 1 root root     5609 Oct 16 10:23 eval_harness.py
-rw------- 1 root root 13672932 Oct 16 10:23 m1_650_preds.csv
-rw------- 1 root root 12953846 Oct 16 10:23 m2_preds.csv
-rw------- 1 root root 57220648 Oct 16 10:23 pairs_3d.csv
-rw------- 1 root root 53544459 Oct 16 10:23 pairs.csv
-rw------- 1 root root    10097 Oct 16 10:23 sequences.csv
-rw------- 1 root root    10617 Oct 16 10:23 train_m1.py
-rw------- 1 root root     9533 Oct 16 10:23 train_m2.py
-rw------- 1 root root 40442997 Oct 16 10:23 wt_emb_650.pt


In [ ]:
for s in [0, 1, 2]:
    !python3 train_m1.py pairs.csv sequences.csv --esm-model esm2_t33_650M_UR50D --cache wt_emb_650.pt --seed {s} --out m1reg_s{s}.csv

Loading cached WT embeddings from wt_emb_650.pt
Embedding dim = 1280; building features for 235355 pairs ...
fold 0: trained on 188496, predicted 46859  (final train MSE 0.135)
fold 1: trained on 187989, predicted 47366  (final train MSE 0.138)
fold 2: trained on 188446, predicted 46909  (final train MSE 0.147)
fold 3: trained on 187954, predicted 47401  (final train MSE 0.115)
fold 4: trained on 188535, predicted 46820  (final train MSE 0.141)

Wrote m1reg_s0.csv. Now score it:
  python3 eval_harness.py pairs.csv --pred m1reg_s0.csv --name M1_ESM
Loading cached WT embeddings from wt_emb_650.pt
Embedding dim = 1280; building features for 235355 pairs ...
fold 0: trained on 188496, predicted 46859  (final train MSE 0.143)
fold 1: trained on 187989, predicted 47366  (final train MSE 0.139)
fold 2: trained on 188446, predicted 46909  (final train MSE 0.151)
fold 3: trained on 187954, predicted 47401  (final train MSE 0.119)
fold 4: trained on 188535, predicted 46820  (final train MSE 0.14

In [ ]:
for s in [1, 2]:
    !python3 train_m2.py pairs_3d.csv sequences.csv contacts.npz --cache wt_emb_650.pt --seed {s} --out m2_s{s}.csv

M2 covers 221,340 pairs across 47 proteins (those with structure + embeddings).
fold 0: train 174,481  test 46,859  (final train MSE 0.141)
fold 1: train 173,974  test 47,366  (final train MSE 0.152)
fold 2: train 174,431  test 46,909  (final train MSE 0.151)
fold 3: train 173,939  test 47,401  (final train MSE 0.127)
fold 4: train 188,535  test 32,805  (final train MSE 0.159)

Wrote m2_s1.csv and pairs_struct.csv. Compare on the SAME subset:
  python3 eval_harness.py pairs_struct.csv --pred m2_s1.csv --name M2_GNN
  python3 eval_harness.py pairs_struct.csv --pred m1_preds.csv --name M1_same_subset
  python3 eval_harness.py pairs_struct.csv   # M0 floor on same subset
M2 covers 221,340 pairs across 47 proteins (those with structure + embeddings).
fold 0: train 174,481  test 46,859  (final train MSE 0.138)
fold 1: train 173,974  test 47,366  (final train MSE 0.153)
fold 2: train 174,431  test 46,909  (final train MSE 0.152)
fold 3: train 173,939  test 47,401  (final train MSE 0.129)
fol

In [ ]:
!python3 eval_harness.py pairs.csv --pred m1reg_s0.csv --name M1reg_s0


=== M1reg_s0  (held-out-protein folds already baked into pairs.csv) ===
pairs evaluated: 235355   assays: 48
MACRO  (mean over assays): z_Spearman=0.787  z_RMSE=0.522  epi_Spearman=0.379  epi_AUROC=0.660
POOLED (all rows)        : z_Spearman=0.852  z_RMSE=0.555  epi_Spearman=0.267  epi_AUROC=0.688

By selection type (macro epi_Spearman):
  Activity             assays= 4  epi_Spearman=0.198
  Binding              assays= 4  epi_Spearman=0.243
  Expression           assays= 1  epi_Spearman=0.182
  OrganismalFitness    assays= 5  epi_Spearman=0.288
  Stability            assays=34  epi_Spearman=0.435

Per-assay metrics written to eval_per_assay.csv


In [ ]:
# matched comparison: 3 seeds each
!python3 compare_ci.py pairs_struct.csv --m1 m1reg_s0.csv m1reg_s1.csv m1reg_s2.csv --m2 m2_preds.csv m2_s1.csv m2_s2.csv --out ci_matched.tsv
# the comparison currently in the report (original M1, one seed each), for reference
!python3 compare_ci.py pairs_struct.csv --m1 m1_650_preds.csv --m2 m2_preds.csv --out ci_original.tsv

pairs scored by every file: 221,340 in 45 proteins
  bootstrap 100/500
  bootstrap 200/500
  bootstrap 300/500
  bootstrap 400/500
  bootstrap 500/500

M1 files: 3, M2 files: 3; 500 protein resamples
                                metric    M1    M2  M2_minus_M1  ci_low  ci_high  share_boot_M2_better M1_seed_range M2_seed_range
              macro epistasis Spearman 0.384 0.378       -0.005  -0.032    0.022                 0.336   0.379-0.389   0.373-0.382
                 macro epistasis AUROC 0.657 0.642       -0.015  -0.034    0.004                 0.062   0.654-0.663   0.640-0.644
            pooled Spearman: ALL pairs 0.274 0.284        0.010  -0.018    0.038                 0.774   0.271-0.280   0.282-0.287
pooled Spearman: seq-near / 3D-distant 0.333 0.353        0.019  -0.051    0.052                 0.826   0.328-0.341   0.334-0.368
pooled Spearman: seq-near / 3D-contact 0.291 0.273       -0.018  -0.065    0.016                 0.202   0.287-0.295   0.266-0.278
 pooled Spearm

In [ ]:
!cp m1reg_s*.csv m2_s*.csv ci_*.tsv /content/drive/MyDrive/bs6204/